# Aligning Qwen3 with DPO, QLoRA, and PiSSA

> This notebook uses preference pairs from `synthetic_data_dpo.csv` to train `Qwen/Qwen3-8B` with Direct Preference Optimization (DPO). QLoRA keeps the base model in 4-bit precision, while PiSSA provides a data-free initialization for the trainable LoRA adapters.

The goal is to teach the model to prefer helpful Taco Alley customer-service responses over poor responses without running a separate supervised fine-tuning stage.

<a href="https://colab.research.google.com/github/ned1313/Fine-tuning-and-Optimizing-Small-Language-Models/blob/main/notebooks/dpo_qlora_run.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Install the training libraries

> Google Colab already includes PyTorch. This cell installs or updates the Hugging Face libraries used by the notebook.

In [ ]:
import subprocess
import sys

if "google.colab" in sys.modules:
    packages = [
        "transformers",
        "trl",
        "peft",
        "accelerate",
        "bitsandbytes",
        "datasets",
    ]
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-qU", *packages])
    print("Hugging Face training libraries are ready.")
else:
    print("Using the packages from the local environment.")

## Load the preference dataset

> Each row contains a customer `message`, a preferred response in `chosen`, and a lower-quality response in `rejected`. The CSV `prompt` column was used to generate the candidate responses, so it is not part of DPO training.

In [ ]:
from pathlib import Path

from datasets import load_dataset

SEED = 42
BASE_MODEL = "Qwen/Qwen3-8B"
REPO_ROOT = Path.cwd().resolve().parent
DATASETS_DIR = REPO_ROOT / "datasets"
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
RUN_DIR = ARTIFACTS_DIR / "dpo_qlora_run"
DATA_PATH = DATASETS_DIR / "synthetic_data_dpo.csv"

raw_dataset = load_dataset("csv", data_files=str(DATA_PATH), split="train")
raw_dataset = raw_dataset.select_columns(["message", "chosen", "rejected"])
dataset = raw_dataset.train_test_split(test_size=0.15, seed=SEED)

print(f"Training rows: {len(dataset['train'])}")
print(f"Evaluation rows: {len(dataset['test'])}")

In [ ]:
sample = dataset["train"][0]
print("CUSTOMER MESSAGE:\n", sample["message"])
print("\nCHOSEN RESPONSE:\n", sample["chosen"])
print("\nREJECTED RESPONSE:\n", sample["rejected"])

## Apply the Qwen chat template

> The prompt is formatted as a user turn ending at the assistant-generation marker. The two candidate completions remain separate because DPO compares their log probabilities. Qwen's thinking mode is disabled because the dataset contains direct customer-service responses.

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def format_preference_split(split):
    rows = []
    for example in split:
        messages = [{"role": "user", "content": example["message"]}]
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        rows.append({
            "prompt": prompt,
            "chosen": example["chosen"] + tokenizer.eos_token,
            "rejected": example["rejected"] + tokenizer.eos_token,
        })
    return split.from_list(rows)

train_dataset = format_preference_split(dataset["train"])
eval_dataset = format_preference_split(dataset["test"])

In [ ]:
formatted_sample = train_dataset[0]
print(formatted_sample["prompt"])
print("Prompt tokens:", len(tokenizer(formatted_sample["prompt"]).input_ids))
print("Chosen tokens:", len(tokenizer(formatted_sample["chosen"]).input_ids))
print("Rejected tokens:", len(tokenizer(formatted_sample["rejected"]).input_ids))

## Load Qwen3 for PiSSA initialization

> PiSSA must inspect floating-point weights before quantization. Load the base model in `bfloat16` or `float16` on CPU for this one-time initialization. Qwen3-8B requires a high-RAM Colab runtime and about 16 GB of temporary disk for the residual model.

In [ ]:
import torch
from transformers import AutoModelForCausalLM

compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Loading Qwen3-8B in floating-point precision on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    dtype=compute_dtype,
    device_map="cpu",
    low_cpu_mem_usage=True,
)
model.config.use_cache = False
print(f"Model footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

## Initialize and save PiSSA

> PiSSA decomposes each targeted floating-point weight into a frozen residual and trainable adapter matrices. Save both pieces before releasing the floating-point model so the residual can be reloaded in 4-bit precision.

In [ ]:
import gc

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules="all-linear",
    task_type="CAUSAL_LM",
    init_lora_weights="pissa",
)

print("Initializing PiSSA adapters. Exact SVD may take several minutes...")
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

pissa_dir = RUN_DIR / "pissa_init"
adapter_dir = pissa_dir / "adapter"
residual_dir = pissa_dir / "residual_model"

model.save_pretrained(adapter_dir)
residual_model = model.unload()
residual_model.save_pretrained(
    residual_dir,
    safe_serialization=True,
    max_shard_size="4GB",
)
print(f"Saved PiSSA initialization to {pissa_dir.resolve()}")

del model
del residual_model
gc.collect()

## Reload the PiSSA residual with QLoRA

> The saved residual now becomes the frozen 4-bit base model. The saved PiSSA adapter is attached separately and remains trainable for DPO.

In [ ]:
from peft import LoraConfig, PeftModel, prepare_model_for_kbit_training
from transformers import BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

print("Reloading the PiSSA residual in 4-bit precision...")
base_model = AutoModelForCausalLM.from_pretrained(
    residual_dir,
    quantization_config=quantization_config,
    device_map="auto",
    dtype=compute_dtype,
)
base_model = prepare_model_for_kbit_training(
    base_model,
    use_gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
)

reload_config = LoraConfig.from_pretrained(adapter_dir)
reload_config.init_lora_weights = True
model = PeftModel.from_pretrained(
    base_model,
    adapter_dir,
    config=reload_config,
    is_trainable=True,
)
model.config.use_cache = False
model.print_trainable_parameters()
print(f"QLoRA model footprint: {model.get_memory_footprint() / 1024**3:.2f} GB")

## Configure Direct Preference Optimization

> `beta` controls how strongly the trained policy is constrained to the reference policy. TRL preserves the initial PiSSA adapter as a frozen `ref` adapter. Reference log probabilities are computed once and cached before each dataset's first evaluation or training pass instead of being recomputed every step.

In [ ]:
from trl import DPOConfig, DPOTrainer

training_args = DPOConfig(
    output_dir=str(RUN_DIR),
    max_length=512,
    beta=0.1,
    loss_type="sigmoid",
    num_train_epochs=1,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    precompute_ref_log_probs=True,
    precompute_ref_batch_size=1,
    learning_rate=5e-6,
    lr_scheduler_type="cosine",
    warmup_steps=2,
    eval_strategy="steps",
    eval_steps=10,
    save_strategy="steps",
    save_steps=10,
    logging_steps=1,
    bf16=(compute_dtype == torch.bfloat16),
    fp16=(compute_dtype == torch.float16),
    seed=SEED,
    report_to="none",
)

trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)
print("Adapters available:", list(trainer.model.peft_config))

## Measure the starting preference accuracy

> DPO reward accuracy is the fraction of pairs where the policy improves the chosen response more than the rejected response relative to the frozen reference policy. The first evaluation also caches the evaluation set's reference log probabilities.

In [ ]:
print("Evaluating baseline reward accuracy before training...")
baseline_metrics = trainer.evaluate()
print(f"Baseline reward accuracy: {baseline_metrics['eval_rewards/accuracies']:.3f}")

## Run DPO training

> During training, watch the loss and reward-accuracy metrics. Accuracy moving above 0.5 indicates that the policy is learning to rank preferred responses higher than rejected responses.

In [ ]:
import time

print("Starting DPO training...")
started = time.perf_counter()
train_result = trainer.train()
elapsed_minutes = (time.perf_counter() - started) / 60

print(f"Training completed in {elapsed_minutes:.1f} minutes.")
print(f"Final training loss: {train_result.training_loss:.4f}")

## Compare evaluation metrics

> A useful DPO run should improve preference accuracy while keeping the policy close enough to the frozen reference policy to avoid unstable updates.

In [ ]:
print("Evaluating the trained policy...")
final_metrics = trainer.evaluate()

baseline_accuracy = baseline_metrics["eval_rewards/accuracies"]
final_accuracy = final_metrics["eval_rewards/accuracies"]

print(f"Baseline reward accuracy: {baseline_accuracy:.3f}")
print(f"Final reward accuracy:    {final_accuracy:.3f}")
print(f"Change:                   {final_accuracy - baseline_accuracy:+.3f}")

In [ ]:
import matplotlib.pyplot as plt

history = trainer.state.log_history
steps = [entry["step"] for entry in history if "loss" in entry]
losses = [entry["loss"] for entry in history if "loss" in entry]

plt.figure(figsize=(8, 4))
plt.plot(steps, losses, marker="o")
plt.xlabel("Optimizer step")
plt.ylabel("DPO loss")
plt.title("DPO training loss")
plt.grid(alpha=0.3)
plt.show()

## Save the trained adapter

> QLoRA saves only the adapter weights and tokenizer files. The original Qwen3-8B model is still required when the adapter is loaded for inference.

In [ ]:
from pathlib import Path

adapter_dir = Path(training_args.output_dir) / "final_adapter"
trainer.save_model(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"Saved the DPO adapter to {adapter_dir.resolve()}")